Fine-Tuning llama3 model on chain of thought grade 5 mathematics dataset

Installation

In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [2]:
# !pip install unsloth accelerate peft datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.3/128.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import unsloth
import torch
from datasets import load_dataset,DatasetDict
from unsloth import FastLanguageModel

max_seq_len = 2048
dtype = None
load_in_4bit = True

model,tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-3B-Instruct',
    max_seq_length=max_seq_len,
    dtype=dtype,
    load_in_4bit=load_in_4bit
)

==((====))==  Unsloth 2025.6.2: Fast Llama patching. Transformers: 4.52.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.11/dist-packages/transformers/quantizers/auto.py:222: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

Now adding LoRA adapters so we only need to update 1 to 10% of all parameters!

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.6.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [11]:
dataset = load_dataset("Adarsh203/math_cot_dataset_new",split="train")

README.md:   0%|          | 0.00/533 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/19.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/80 [00:00<?, ? examples/s]

In [17]:
dataset[0]['conversations']

[{'from': 'user',
  'value': 'Question: Sarah has 3 boxes of cookies. Each box has 12 cookies. She wants to share them equally among herself and 5 friends. How many cookies will each person get?'},
 {'from': 'assistant',
  'value': "Thought: First, I need to find the total number of cookies Sarah has. She has 3 boxes, and each box has 12 cookies. So, I need to multiply the number of boxes by the number of cookies in each box.\nThought: Total cookies = 3 * 12 = 36 cookies.\nThought: Next, I need to find out how many people will be sharing the cookies. It's Sarah herself and 5 friends, so that's 1 (Sarah) + 5 (friends) = 6 people.\nThought: Finally, to find out how many cookies each person gets, I need to divide the total number of cookies by the total number of people.\nThought: Cookies per person = 36 / 6 = 6 cookies.\nAnswer: Each person will get 6 cookies."}]

In [9]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1"
)

In [18]:
def formatting_prompts_func(examples):
  convos = examples['conversations']
  texts = [tokenizer.apply_chat_template(convo,tokenize=False,add_generation_prompt = False) for convo in convos]
  return {"text": texts}


We now use `standardize_sharegpt` to convert ShareGPT style datasets into HuggingFace's generic format. This changes the dataset from looking like:
```
{"from": "system", "value": "You are an assistant"}
{"from": "human", "value": "What is 2+2?"}
{"from": "gpt", "value": "It's 4."}
```
to
```
{"role": "system", "content": "You are an assistant"}
{"role": "user", "content": "What is 2+2?"}
{"role": "assistant", "content": "It's 4."}
```

In [19]:
from unsloth.chat_templates import standardize_sharegpt

dataset = standardize_sharegpt(dataset)
dataset = dataset.map(formatting_prompts_func,batched=True)

Unsloth: Standardizing formats (num_proc=2):   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

In [20]:
dataset[5]["conversations"]

[{'content': 'Question: A factory produces 2,345 toys in one week. How many toys will it produce in 3 weeks?',
  'role': 'user'},
 {'content': 'Thought: To find the total number of toys produced in 3 weeks, I need to multiply the number of toys produced in one week by 3.\nThought: Toys in 3 weeks = 2,345 * 3.\nThought: I will multiply 2345 by 3.\nThought: 5 * 3 = 15 (write down 5, carry over 1).\nThought: 4 * 3 = 12 + 1 (carried over) = 13 (write down 3, carry over 1).\nThought: 3 * 3 = 9 + 1 (carried over) = 10 (write down 0, carry over 1).\nThought: 2 * 3 = 6 + 1 (carried over) = 7.\nThought: So, 2,345 * 3 = 7,035.\nAnswer: The factory will produce 7,035 toys in 3 weeks.',
  'role': 'assistant'}]

In [21]:
dataset[5]["text"]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nQuestion: A factory produces 2,345 toys in one week. How many toys will it produce in 3 weeks?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nThought: To find the total number of toys produced in 3 weeks, I need to multiply the number of toys produced in one week by 3.\nThought: Toys in 3 weeks = 2,345 * 3.\nThought: I will multiply 2345 by 3.\nThought: 5 * 3 = 15 (write down 5, carry over 1).\nThought: 4 * 3 = 12 + 1 (carried over) = 13 (write down 3, carry over 1).\nThought: 3 * 3 = 9 + 1 (carried over) = 10 (write down 0, carry over 1).\nThought: 2 * 3 = 6 + 1 (carried over) = 7.\nThought: So, 2,345 * 3 = 7,035.\nAnswer: The factory will produce 7,035 toys in 3 weeks.<|eot_id|>'

Training the Model

In [24]:
from trl import SFTTrainer
from transformers import TrainingArguments,DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_len,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        # max_steps = None,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"]:   0%|          | 0/80 [00:00<?, ? examples/s]

We also use Unsloth's `train_on_completions` method to only train on the

---

assistant outputs and ignore the loss on the user's inputs.

In [25]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map (num_proc=2):   0%|          | 0/80 [00:00<?, ? examples/s]

In [26]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

'<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nQuestion: A factory produces 2,345 toys in one week. How many toys will it produce in 3 weeks?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nThought: To find the total number of toys produced in 3 weeks, I need to multiply the number of toys produced in one week by 3.\nThought: Toys in 3 weeks = 2,345 * 3.\nThought: I will multiply 2345 by 3.\nThought: 5 * 3 = 15 (write down 5, carry over 1).\nThought: 4 * 3 = 12 + 1 (carried over) = 13 (write down 3, carry over 1).\nThought: 3 * 3 = 9 + 1 (carried over) = 10 (write down 0, carry over 1).\nThought: 2 * 3 = 6 + 1 (carried over) = 7.\nThought: So, 2,345 * 3 = 7,035.\nAnswer: The factory will produce 7,035 toys in 3 weeks.<|eot_id|>'

In [29]:
space = tokenizer(" ",add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

'                                                             Thought: To find the total number of toys produced in 3 weeks, I need to multiply the number of toys produced in one week by 3.\nThought: Toys in 3 weeks = 2,345 * 3.\nThought: I will multiply 2345 by 3.\nThought: 5 * 3 = 15 (write down 5, carry over 1).\nThought: 4 * 3 = 12 + 1 (carried over) = 13 (write down 3, carry over 1).\nThought: 3 * 3 = 9 + 1 (carried over) = 10 (write down 0, carry over 1).\nThought: 2 * 3 = 6 + 1 (carried over) = 7.\nThought: So, 2,345 * 3 = 7,035.\nAnswer: The factory will produce 7,035 toys in 3 weeks.<|eot_id|>'

In [30]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 80 | Num Epochs = 1 | Total steps = 10
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856/3,000,000,000 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.084800
2,1.125800
3,0.936400
4,0.860700
5,0.801800
6,0.774000
7,0.743900
8,0.634900
9,0.507900
10,0.603500


<a name="Inference"></a>
### Inference
Let's run the model! We can change the instruction and input - leave the output blank!



We use `min_p = 0.1` and `temperature = 1.5`.

In [31]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1"
)
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

<p>[INST]
Question: Ben bought a toy for $14 and a game for $26. He paid with a $50 bill. How much change did he get?
[/INST]
</p>

Thought: First, add the cost of the toy and the game.
Thought: Total cost = $14 + $26 = $40.
Thought: Then subtract from the amount he paid.
Thought: Change = $50 − $40 = $10.
Answer: Ben got $10 in change.

In [32]:
messages = [
    {"role": "user","content": "Question: Ben bought a toy for $14 and a game for $26. He paid with a $50 bill. How much change did he get?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # must add for generation
    return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs,max_new_tokens=256,use_cache=True,temperature=1.5,min_p=0.1)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [33]:
tokenizer.batch_decode(outputs)

["<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nQuestion: Ben bought a toy for $14 and a game for $26. He paid with a $50 bill. How much change did he get?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nThought: First, I'll add up the cost of the toy and the game, which is $14 + $26 = $40.\nThought: Then, I'll subtract this total from the $50 bill, which is $50 - $40 = $10.\nThought: Since Ben got $10 back, I can say that he got $10 in change.\n\nAnswer: Ben got $10 in change.<|eot_id|>"]

In [34]:
from transformers import TextStreamer

text_streamer = TextStreamer(tokenizer,skip_prompt=True)
_ = model.generate(input_ids=inputs,streamer = text_streamer,max_new_tokens=256,use_cache=True,temperature=1.5,min_p=0.1)

Thought: First, let's add the costs of the toys and games Ben bought: $14 + $26 = $40
Thought: Now, let's calculate how much change Ben should get: $50 - $40 = $10
Thought: So Ben got $10 change.
Answer: He got $10 change.<|eot_id|>


Saving finetuned models

In [35]:
model.save_pretrained('lora_model')
tokenizer.save_pretrained("lora_model")

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/chat_template.jinja',
 'lora_model/tokenizer.json')

In [38]:
model.push_to_hub("Adarsh203/Llama-3.2-3B-Instruct_cot_lora_model",token=userdata.get('HUGGINGFACEHUB_API_TOKEN'))
tokenizer.push_to_hub("Adarsh203/Llama-3.2-3B-Instruct_cot_lora_model",token=userdata.get('HUGGINGFACEHUB_API_TOKEN'))

README.md:   0%|          | 0.00/614 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/97.3M [00:00<?, ?B/s]

Saved model to https://huggingface.co/Adarsh203/Llama-3.2-3B-Instruct_cot_lora_model


tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [40]:
## using the saved model on huggingface for inference
if True:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "Adarsh203/Llama-3.2-3B-Instruct_cot_lora_model",
        max_seq_length = max_seq_len,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

==((====))==  Unsloth 2025.6.2: Fast Llama patching. Transformers: 4.52.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


adapter_model.safetensors:   0%|          | 0.00/97.3M [00:00<?, ?B/s]

To find the change Ben got, first, I'll need to find the total cost of the items. The total cost is $14 (toy) + $26 (game) = $40.

Next, I'll need to find the amount he paid with, which is $50.

To find the change, subtract the total cost from the amount he paid: $50 - $40 = $10.

Ben got $10 as change.<|eot_id|>
